# Practice guide 04_03: functionality-preserving evasion and the defence

**Goal.** Attack the PE-header detector of guide 04_01 yourself, three ways: the malleable rewrite, benign
mimicry, and a greedy black-box query attack; measure evasion rates; then implement the adversarial
retraining defence and test whether it generalises to an attack it did not see. Every `TODO` cell must be
completed; the `check` cells tell you whether you got it right.

Read `guide_04_03.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. Setup (given)

The detector of guide 04_01 (recall ≈ .997) and the median benign profile. `mw.MALLEABLE` lists the header
fields an attacker may set freely — rewriting them costs nothing at runtime.

In [ ]:
X, y, cols = mw.load_pe()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
mal = np.where(yte == 1)[0]
med = np.median(Xtr[ytr == 0], 0)
mal_fields = [cols.index(c) for c in mw.MALLEABLE if c in cols]
print(f"baseline malicious recall {recall_score(yte, rf.predict(Xte), pos_label=1):.4f}")
print(f"malleable fields present: {len(mal_fields)} of {len(mw.MALLEABLE)}")


def evasion(model, Xa, mask=mal, thr=0.5):
    """Share of malicious test files scored below thr after the attack."""
    return mw.evasion_rate(model, Xa[mask], np.ones(mask.sum()), thr=thr)

**Question A1.** Why does an attacker prefer functionality-preserving manipulations over, say, appending
bytes that lower the score? State the two constraints every "good" evasion attack must satisfy.

## Part B. Attack A — malleable rewrite

Set every malleable field of every malicious test file to the benign median. Implement it with `mw.median_rewrite`
and measure the evasion rate.

In [ ]:
# TODO: XA = mw.median_rewrite(Xte, Xte, mal_fields, med); evA = evasion(rf, XA); print it.

**Question B1.** Attack A reaches evasion 1.000. What does a perfect evasion rate mean operationally — and
which single number of the *undefended* detector would you quote to argue that the benchmark, not the method,
is what failed?

## Part C. Attack B — benign mimicry

Copy a random benign profile onto each malicious file and keep only `k` of its fields from the payload.
Sweep k = 40, 20, 10, 5 and record the evasion rates.

In [ ]:
rng = np.random.default_rng(0)
evs = {}
# TODO: for k in [40, 20, 10, 5]:
#   XB = mw.benign_template_attack(Xte[mal], Xtr[ytr == 0], keep=k, rng=rng, cols=X.shape[1]);
#   put XB into a copy of Xte at the malicious rows; evs[k] = evasion(rf, XBf). Print the table.

**Question C1.** Evasion *rises* as k falls. Explain the trade-off from the attacker's seat: what does small
k buy, what does it risk, and why is the risk invisible in this experiment?

## Part D. Attack C — greedy black-box

The realistic attack: the attacker only *queries* the detector. Try each malleable field at its benign
median, apply the single rewrite that lowers the score the most, repeat up to 30 steps. Implement the loop
for the first 60 malicious test files (mw.greedy_evasion is the reference; write yours and check it agrees).

In [ ]:
def my_greedy(model, x, candidates, med, max_steps=30):
    x = x.copy()
    # TODO: loop up to max_steps: score x; stop if below 0.5; otherwise for every candidate j compute the
    # score of x with x[j] = med[j]; apply the best j; stop if no candidate lowers the score.
    # Hint: score all candidates in ONE call: stack the rows (np.tile) and use model.predict_proba(rows)[:, 1].
    raise NotImplementedError

In [ ]:
sub = mal[:60]
XC = Xte.copy()
for i in sub:
    XC[i] = my_greedy(rf, Xte[i], mal_fields, med)
evC = evasion(rf, XC, mask=sub)
ref = np.array([mw.greedy_evasion(rf, x, mal_fields, med) for x in Xte[sub]])
assert np.allclose(XC[sub], ref)
print(f"attack C evasion {evC:.3f} (greedy implementation matches the reference)")

**Question D1.** Attack C reaches the same evasion as A with no gradients and no model access — only query access.
Estimate the query budget: how many `predict_proba` calls does one file cost, at most, and what does that
suggest about rate-limiting scanner APIs?

## Part E. The defence: adversarial retraining

Retrain the forest with attack-A-rewritten *training* malware added to the training set. Then evaluate the
defended model on all three attacks.

In [ ]:
rf_def = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
Xadv = {}
# TODO 1: Xadv_A = mw.median_rewrite(Xtr, Xtr, mal_fields, med); build the augmented training set
#         (original rows + Xadv_A rows with ytr==1) and fit rf_def on it.
# TODO 2: defended numbers: baseline recall on Xte, attack A evasion, attack B (k=10) evasion on the defended
#         model, attack C evasion on the defended model (first 60 files). Print all four.

**Question E1.** The defence kills attack A and the greedy attack C (0.02) but mimicry B still evades (0.94).
Why does retraining only patch the region it saw, and what would an *adaptive* attacker do next? Name the arms-race step that ends
this loop for a defender with finite resources.

## Part F. The evasion summary table

Fill the table: evasion of the undefended model (A, B k=10, C) and of the defended model (same three).

In [ ]:
rows = {}
# TODO: rows = {("undefended", "A"): evA, ("undefended", "B"): evs[10], ("undefended", "C"): evC,
#               ("defended", "A"): ..., ("defended", "B"): ..., ("defended", "C"): ...}
#       print them as a table with 2 decimals.

**Question F1.** Which attack is the most robust across both models, and why does that make mimicry the
canonical threat for ML-based malware detectors?

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_03.ipynb`.